# AfriVoices — Réentraînement v3 (fine-tuning depuis v2)

**But** : faire baisser le macro-WER en **augmentant le volume des langues faibles**
(som, kln, mas surtout), sans nouvelles données — on puise simplement plus profond dans
`anv_data_ke` (54k–136k clips dispo par langue, on n'en utilisait que ~1500–2500).

**Principe** : fine-tuning depuis le checkpoint **v2** (on garde l'acquis), avec un
learning rate faible (1e-5) pour ne pas détruire ce que le modèle sait déjà, et un
temperature sampling plus agressif (T=1.5) pour donner plus de poids aux langues rares.

**Sécurité** : on écrit dans `w2vbert-ctc-v3` (dossier distinct). v2 reste intact comme
filet de secours. Les KenLM existants se réappliqueront tels quels sur v3.

**Cible matériel** : GPU modeste (T4/L4). Volume calibré pour tenir en quelques heures.

## 1. Dépendances

In [ ]:
!pip install -q "transformers>=4.44,<5" "datasets>=3.5,<4" "accelerate>=0.33" \
    "jiwer>=3.0" "evaluate>=0.4" torchaudio soundfile ffmpeg-python librosa huggingface_hub
print("ok — redémarre le runtime si demandé, puis reprends à la §2")

## 2. Configuration + connexion

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, numpy as np, torch
from huggingface_hub import login
# login("hf_...")   # décommente et mets ton token si pas déjà connecté

V2_DIR  = "/content/drive/MyDrive/afrivoices/w2vbert-ctc-v2"   # point de départ (lecture)
V3_DIR  = "/content/drive/MyDrive/afrivoices/baobab-asr-v3"   # sortie (écriture)
SAMPLING_RATE = 16_000
MAX_AUDIO_S   = 20
SEED          = 42
TEMPERATURE   = 1.5      # plus haut que v2 (1.3) -> plus de poids aux langues rares

# volumes PAR TYPE (scripted + unscripted) -> total ≈ 2x ces valeurs
# on pousse fort som/kln/mas, on maintient luo/kik, swa géré à part (anti-oubli)
N_PER_TYPE = {"som": 3000, "kln": 3000, "mas": 2500, "luo": 2000, "kik": 1500}
SWA_PER_SHARD = 400      # 5 domaines x 400 ≈ 2000 swahili (juste pour ne pas l'oublier)

LANG_HOURS = {"swa":2979, "som":1002, "kik":754, "luo":723, "kln":521, "mas":505}
LANGS = list(LANG_HOURS.keys())
os.makedirs(V3_DIR, exist_ok=True)
torch.manual_seed(SEED); np.random.seed(SEED)

def temperature_probs(hours, T):
    h = np.array(list(hours.values()), float); w = h**(1.0/T); w/=w.sum()
    return dict(zip(hours.keys(), w))
probs = temperature_probs(LANG_HOURS, TEMPERATURE)
print("probas échantillonnage (T=%.1f):"%TEMPERATURE, {k:round(v,3) for k,v in probs.items()})
print("v2 existe :", os.path.isdir(V2_DIR), "| vocab :", os.path.exists(V2_DIR+"/vocab.json"))

## 3. Charger le processor de v2 (vocab figé — NE PAS régénérer)

Critique pour le fine-tuning : on réutilise **exactement** le vocab.json et le processor
de v2. Régénérer un vocab changerait l'ordre des caractères et casserait la tête CTC.

In [ ]:
import re
from transformers import Wav2Vec2BertProcessor

processor = Wav2Vec2BertProcessor.from_pretrained(V2_DIR)
tokenizer = processor.tokenizer
feature_extractor = processor.feature_extractor
print("vocab v2 chargé, taille:", len(tokenizer))

# clean_text IDENTIQUE à l'entraînement v2 (sinon désalignement texte/modèle)
def clean_text(t):
    t = (t or "").lower().strip()
    t = re.sub(r"[^\w\sĩũáàâäéèêëíìîïóòôöúùûü']", "", t, flags=re.UNICODE)
    t = re.sub(r"\s+", " ", t)
    return t

## 4. Charger les données (mêmes loaders que v2, volumes augmentés)

### Login Huggingface

In [ ]:
from huggingface_hub import login
login("hf_...")  # champ masqué — ne colle jamais ton token en clair

### 4.1 — Swahili (manifests → wav), volume modéré (anti-oubli seulement)

In [ ]:
import json, tarfile, subprocess
from huggingface_hub import hf_hub_download
from datasets import Dataset, Audio

SWA_REPO = "DigitalUmuganda/Afrivoice_Swahili"
SWA_WORK = "/content/swa"; os.makedirs(SWA_WORK, exist_ok=True)
SWA_TEXT = ["normalized_transcription", "transcription"]

def webm_to_wav(src, dst):
    subprocess.run(["ffmpeg","-y","-i",src,"-ar","16000","-ac","1",dst,"-loglevel","error"], check=True)

def load_swahili(domains=("agriculture","health","education","financial","government"),
                 split="train", shards_per_domain=1, max_per_shard=SWA_PER_SHARD):
    rows=[]
    for dom in domains:
        folder=f"{dom}_swahili_{split}"
        for n in range(shards_per_domain):
            try:
                man = hf_hub_download(SWA_REPO, f"{folder}/manifest_{n}.jsonl", repo_type="dataset")
                tarp= hf_hub_download(SWA_REPO, f"{folder}/audio/audio_{n}.tar.xz", repo_type="dataset")
            except Exception as e:
                print("skip",folder,n,str(e)[:60]); continue
            want={}
            with open(man,encoding="utf-8") as f:
                for line in f:
                    if not line.strip(): continue
                    d=json.loads(line)
                    txt=next((d[k] for k in SWA_TEXT if d.get(k)),None)
                    key=d.get("key") or os.path.splitext(os.path.basename(d.get("audio_filepath","")))[0]
                    if txt and key: want[key]=txt.strip()
            ex=os.path.join(SWA_WORK,folder,f"s{n}"); os.makedirs(ex,exist_ok=True); done=0
            with tarfile.open(tarp,"r:xz") as t:
                for m in t:
                    if done>=max_per_shard: break
                    if not m.isfile() or not m.name.endswith(".webm"): continue
                    key=os.path.splitext(os.path.basename(m.name))[0]
                    if key not in want: continue
                    fobj=t.extractfile(m)
                    if fobj is None: continue
                    webm=os.path.join(ex,key+".webm"); wav=os.path.join(ex,key+".wav")
                    with open(webm,"wb") as o: o.write(fobj.read())
                    try: webm_to_wav(webm,wav); os.remove(webm)
                    except Exception: continue
                    rows.append({"audio":wav,"text":want[key],"lang":"swa"}); done+=1
            print(f"{folder} s{n}: +{done} (total {len(rows)})")
    return Dataset.from_list(rows).cast_column("audio", Audio(sampling_rate=16000))

swa = load_swahili()
print(swa)

### 4.2 — 5 langues kényanes (parquet/WAV), volumes augmentés

`decode` direct des bytes WAV (l'entraînement est en bytes bruts, pas base64 — confirmé).
`n_per_type` est **par type** : som=3000 → jusqu'à 6000 som (scripted + unscripted).

In [ ]:
import soundfile as sf, io, base64
from datasets import load_dataset

KE_REPO  = "MCAA1-MSU/anv_data_ke"
KE_LANGS = ["kik","luo","kln","mas","som"]
KE_OUT   = "/content/ke_wav"; os.makedirs(KE_OUT, exist_ok=True)

def _decode(a):
    b=a.get("bytes")
    if b:
        try: arr,sr=sf.read(io.BytesIO(b),dtype="float32")
        except Exception: arr,sr=sf.read(io.BytesIO(base64.b64decode(b)),dtype="float32")
        return arr,sr
    return a["array"], a["sampling_rate"]

def load_ke(n_per_type, types=("unscripted","scripted")):
    paths,texts,langs=[],[],[]
    for code in KE_LANGS:
        target = n_per_type.get(code,0)
        if target<=0: continue
        for typ in types:
            patt=f"hf://datasets/{KE_REPO}/{code}/train/{typ}/audios/*.parquet"
            try:
                ds=load_dataset("parquet", data_files=patt, split="train", streaming=True)
            except Exception as e:
                print("skip",code,typ,str(e)[:60]); continue
            it=iter(ds); got=0; bad=0
            while got<target:
                try: ex=next(it)
                except StopIteration: break
                except Exception:
                    bad+=1
                    if bad>50: break
                    continue
                try:
                    txt=(ex.get("transcription") or "").strip()
                    if not txt: continue
                    arr,sr=_decode(ex["audio"])
                    fp=os.path.join(KE_OUT,f"{code}_{typ}_{got}.wav")
                    sf.write(fp,arr,sr)
                    paths.append(fp); texts.append(txt); langs.append(code); got+=1
                except Exception:
                    bad+=1; continue
            print(f"{code}/{typ}: +{got} (sautés {bad}, total {len(paths)})")
    from datasets import Dataset, Audio
    return Dataset.from_dict({"audio":paths,"text":texts,"lang":langs}).cast_column("audio", Audio(sampling_rate=16000))

ke = load_ke(N_PER_TYPE)
print(ke)

### 4.3 — Regrouper par langue

In [ ]:
datasets_norm = {"swa": swa}
for code in ["kik","luo","kln","mas","som"]:
    datasets_norm[code] = ke.filter(lambda x, c=code: x["lang"]==c)
for l,d in datasets_norm.items(): print(l, len(d))

## 5. Préparation (features + labels) avec le tokenizer de v2

In [ ]:
from datasets import disable_caching, enable_caching
enable_caching()   # IMPORTANT : laisse datasets écrire le cache sur disque (pas en RAM)

def prepare(batch):
    audio = batch["audio"]
    batch["input_features"] = feature_extractor(audio["array"], sampling_rate=SAMPLING_RATE).input_features[0]
    batch["input_length"] = len(audio["array"])/SAMPLING_RATE
    batch["labels"] = tokenizer(clean_text(batch["text"])).input_ids
    return batch

prepared = {}
for l, d in datasets_norm.items():
    d = d.filter(lambda ex: ex["text"] and len(ex["text"].strip())>0)
    d = d.map(
        prepare,
        remove_columns=d.column_names,
        num_proc=1,              # 1 seul process -> pas de duplication mémoire
        writer_batch_size=200,   # écrit sur disque tous les 200 ex (défaut 1000 = trop gros)
        keep_in_memory=False,    # force l'écriture disque, pas la RAM
    )
    d = d.filter(lambda ln: ln < MAX_AUDIO_S, input_columns=["input_length"])
    prepared[l] = d
    print(l, len(d), "OK")
    import gc; gc.collect()

## 6. Mélange multilingue (T=1.5) + split eval par langue

In [ ]:
from datasets import interleave_datasets
eval_per_lang={}; train_parts=[]
for l in LANGS:
    if l not in prepared: continue
    sp=prepared[l].train_test_split(test_size=0.03, seed=SEED)
    train_parts.append(sp["train"]); eval_per_lang[l]=sp["test"]
order=[l for l in LANGS if l in prepared]
train_ds=interleave_datasets(train_parts, probabilities=[probs[l] for l in order],
                             seed=SEED, stopping_strategy="all_exhausted")
print("train:", train_ds)
for l,d in eval_per_lang.items(): print("eval",l,len(d))

## 6bis. 💾 Sauvegarde des données préparées (à faire en CPU)

**Économie d'unités GPU** : on prépare tout en runtime **CPU** (gratuit), on sauvegarde
sur le Drive, puis on bascule en GPU et on recharge sans repayer la préparation.

Exécute §1→§6bis en **CPU**. Ensuite : Exécution → Modifier le type d'exécution → GPU.

In [ ]:
import os
SAVE_DIR = "/content/drive/MyDrive/afrivoices/prepared_v3"
os.makedirs(SAVE_DIR, exist_ok=True)

train_ds.save_to_disk(f"{SAVE_DIR}/train")
import json
# eval : on sauve chaque langue séparément pour garder le WER par langue
os.makedirs(f"{SAVE_DIR}/eval", exist_ok=True)
for l, d in eval_per_lang.items():
    d.save_to_disk(f"{SAVE_DIR}/eval/{l}")
with open(f"{SAVE_DIR}/langs.json","w") as f:
    json.dump(list(eval_per_lang.keys()), f)

print("✅ données préparées sauvegardées ->", SAVE_DIR)
print("Maintenant : passe en runtime GPU, puis exécute la §7bis (rechargement) avant la §8.")

## 7. Collator CTC

In [ ]:
from dataclasses import dataclass
from typing import List, Dict, Union
import torch

@dataclass
class DataCollatorCTCWithPadding:
    processor: object
    def __call__(self, features):
        input_features=[{"input_features":f["input_features"]} for f in features]
        label_features=[{"input_ids":f["labels"]} for f in features]
        batch=self.processor.feature_extractor.pad(input_features, padding=True, return_tensors="pt")
        lb=self.processor.tokenizer.pad(label_features, padding=True, return_tensors="pt")
        batch["labels"]=lb["input_ids"].masked_fill(lb.attention_mask.ne(1), -100)
        return batch
data_collator=DataCollatorCTCWithPadding(processor=processor)

## 7bis. 📂 Rechargement après bascule GPU (à faire en GPU)

Après être passé en runtime GPU : recharge ici les données préparées (rapide, pas de
re-téléchargement). Puis exécute la §7 (collator) et la §8 (entraînement).

⚠️ En GPU, ré-exécute d'abord §1 (deps), §2 (config+Drive) et §3 (processor) — le kernel
a redémarré, ces variables doivent être recréées. Tu peux **sauter §4, §5, §6, §6bis**
(c'est ce que ce rechargement remplace).

In [ ]:
import json
from datasets import load_from_disk
SAVE_DIR = "/content/drive/MyDrive/afrivoices/prepared_v3"

train_ds = load_from_disk(f"{SAVE_DIR}/train")
langs = json.load(open(f"{SAVE_DIR}/langs.json"))
eval_per_lang = {l: load_from_disk(f"{SAVE_DIR}/eval/{l}") for l in langs}

print("✅ rechargé | train:", len(train_ds), "| eval:", {l: len(d) for l,d in eval_per_lang.items()})

## 8. Fine-tuning depuis v2 → v3

Différences clés avec l'entraînement initial :
- modèle chargé depuis **V2_DIR** (pas la base) → on continue, on ne repart pas de zéro,
- **learning rate 1e-5** (5x plus bas) → fine-tuning doux, préserve l'acquis,
- sortie **V3_DIR** distinct → v2 préservé,
- batch petit + checkpointing pour tenir sur T4/L4.

In [ ]:
from transformers import Wav2Vec2BertForCTC, TrainingArguments, Trainer
from datasets import concatenate_datasets
import evaluate
wer_metric = evaluate.load("wer")

# charger les POIDS de v2 (la tête CTC + le vocab sont déjà bons)
model = Wav2Vec2BertForCTC.from_pretrained(V2_DIR, ctc_zero_infinity=True)
model.config.use_cache = False
n = sum(p.numel() for p in model.parameters())
print(f"Params: {n/1e6:.1f} M ({'OK <1Md' if n<1e9 else 'TROP GROS'}) — départ depuis v2")

def preprocess_logits_for_metrics(logits, labels): return logits.argmax(dim=-1)
def compute_metrics(pred):
    ids=pred.predictions; labels=pred.label_ids
    labels[labels==-100]=processor.tokenizer.pad_token_id
    return {"wer": wer_metric.compute(predictions=processor.batch_decode(ids),
                                      references=processor.batch_decode(labels, group_tokens=False))}

eval_all = concatenate_datasets(list(eval_per_lang.values()))
cuda=torch.cuda.is_available()
gpu_gb=torch.cuda.get_device_properties(0).total_memory/1e9 if cuda else 0
big=gpu_gb>30
train_bs=8 if big else 2
accum=2 if big else 8
use_bf16=cuda and torch.cuda.is_bf16_supported()
print(f"GPU ~{gpu_gb:.0f}Go | batch={train_bs} accum={accum} bf16={use_bf16}")

args=TrainingArguments(
    output_dir=V3_DIR,
    per_device_train_batch_size=train_bs, gradient_accumulation_steps=accum,
    per_device_eval_batch_size=train_bs, eval_accumulation_steps=4,
    gradient_checkpointing=not big, gradient_checkpointing_kwargs={"use_reentrant":False},
    learning_rate=1e-5,                 # fine-tuning doux
    warmup_ratio=0.1, num_train_epochs=3,
    bf16=use_bf16, fp16=(cuda and not use_bf16),
    eval_strategy="steps", eval_steps=500, save_steps=500, save_total_limit=2,
    logging_steps=50, load_best_model_at_end=True,
    metric_for_best_model="wer", greater_is_better=False,
    report_to="none", seed=SEED)

trainer=Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=eval_all,
                data_collator=data_collator, compute_metrics=compute_metrics,
                preprocess_logits_for_metrics=preprocess_logits_for_metrics,
                processing_class=processor)

from transformers.trainer_utils import get_last_checkpoint
last = get_last_checkpoint(V3_DIR) if os.path.isdir(V3_DIR) else None
trainer.train(resume_from_checkpoint=last)
trainer.save_model(V3_DIR); processor.save_pretrained(V3_DIR)
print("v3 sauvegardé ->", V3_DIR)

## 9. WER par langue (comparer v3 à v2)

La boussole : on veut voir kln/mas/som baisser sans que swa/luo/kik régressent.

In [ ]:
import torch, numpy as np

@torch.no_grad()
def wer_par_langue(model, eval_per_lang, processor, bs=8):
    """Passe d'inférence par langue sur le split eval -> WER de chaque langue + macro."""
    import evaluate
    wer = evaluate.load("wer")
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    model.eval().to(dev)
    res = {}
    for l, d in eval_per_lang.items():
        preds, refs = [], []
        for i in range(0, len(d), bs):
            chunk = d[i:i+bs]
            feats = processor.feature_extractor.pad(
                [{"input_features": f} for f in chunk["input_features"]],
                padding=True, return_tensors="pt")
            feats = {k: v.to(dev) for k, v in feats.items()}
            logits = model(**feats).logits
            pred_ids = torch.argmax(logits, dim=-1)
            preds += processor.batch_decode(pred_ids)
            labels = [[t for t in lab if t != -100] for lab in chunk["labels"]]
            refs += processor.batch_decode(labels, group_tokens=False)
        res[l] = wer.compute(predictions=preds, references=refs)
        print(f"  {l}: WER {res[l]:.3f}  (n={len(d)})")
    macro = sum(res.values())/len(res)
    print(f"\nMacro-WER v3 : {macro:.3f}")
    return res, macro

print("WER par langue — modèle v3 (eval split) :")
res_v3, macro_v3 = wer_par_langue(model, eval_per_lang, processor)
print("\nRappel v2 (macro 0.474) : swa .131 | luo .287 | kik .336 | mas .622 | kln .750 | som .719")
print("Best WER agrégé (Trainer) :", trainer.state.best_metric)

## 10. Étapes suivantes

1. **Comparer** le WER eval de v3 à celui de v2 (0.474 macro). Si v3 est meilleur :
2. **Re-générer submission** avec v3 + les KenLM existants (notebook principal §13, en
   pointant `OUTPUT_DIR` sur `w2vbert-ctc-v3`).
3. **Soumettre** et comparer le score public à 0.469.
4. Si v3 régresse sur swa/luo/kik (oubli), baisser encore le LR ou augmenter la part swa.

⚠️ Garde v2 : si v3 est moins bon, tu reviens à v2 sans rien perdre.

## 11. Inférence v3 + KenLM → submission

⚠️ **Règles d'or environnement Colab (pour éviter de tout casser)** :
- on installe **uniquement** `kenlm` + `pyctcdecode`, rien d'autre,
- on ne **force JAMAIS** la réinstallation de `torch` (build CUDA de Colab → casse),
- on ne **force JAMAIS** `numpy<2` (Colab utilise numpy 2.x),
- `kenlm` doit être installé **puis le runtime redémarré** avant que `pyctcdecode`
  puisse l'importer.

### 11.1 — Installer kenlm + pyctcdecode (puis REDÉMARRER le runtime)

In [ ]:
!pip install -q pyctcdecode
!pip install -q https://github.com/kpu/kenlm/archive/master.zip
print("Installé. REDÉMARRE le runtime (Exécution → Redémarrer la session), puis §11.2")

### 11.2 — Vérifier les imports (sans rien réinstaller)

In [ ]:
import torch, numpy, kenlm
from transformers import Wav2Vec2BertForCTC, Wav2Vec2BertProcessor
from pyctcdecode import build_ctcdecoder
print("torch", torch.__version__, "| numpy", numpy.__version__)
print("cuda:", torch.cuda.is_available(), "| imports OK")
# Si erreur "numpy.dtype size changed" -> !pip install -q "numpy>=2" --force-reinstall puis redémarrer
# Si erreur torch/_dynamo -> supprimer le runtime (Déconnecter et supprimer) et reprendre §11.1

### 11.3 — Recharger TOUT (modèle v3 + labels + decode_robuste + décodeurs)

Bloc unique et autonome. Le modèle v3 a été sauvé sous `baobab-asr-v3`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import torch, io, base64, os, numpy as np, soundfile as sf, librosa
from transformers import Wav2Vec2BertForCTC, Wav2Vec2BertProcessor
from pyctcdecode import build_ctcdecoder

V3 = "/content/drive/MyDrive/afrivoices/baobab-asr-v3"
LM = "/content/drive/MyDrive/afrivoices/lm"
device = "cuda" if torch.cuda.is_available() else "cpu"

# modèle v3
model = Wav2Vec2BertForCTC.from_pretrained(V3, local_files_only=True).to(device).eval()
processor = Wav2Vec2BertProcessor.from_pretrained(V3, local_files_only=True)

# labels (un seul blank, un seul espace, placeholders uniques)
raw = [t for t,_ in sorted(processor.tokenizer.get_vocab().items(), key=lambda kv: kv[1])]
blank_tok = processor.tokenizer.pad_token
labels, n = [], 0
for t in raw:
    if t == blank_tok: labels.append("")
    elif t in ("|"," "): labels.append(" ")
    elif t in {"[UNK]","<s>","</s>"}: n+=1; labels.append("⁇"*n)
    else: labels.append(t)
assert labels.count("")==1 and labels.count(" ")==1
print("labels OK:", len(labels))

# decode_robuste (bytes bruts OU base64 OU ffmpeg)
def decode_robuste(a):
    b = a.get("bytes")
    if not b: raise ValueError("vide")
    try: arr, sr = sf.read(io.BytesIO(b), dtype="float32")
    except Exception:
        try: arr, sr = sf.read(io.BytesIO(base64.b64decode(b)), dtype="float32")
        except Exception:
            import tempfile
            with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as t:
                try: t.write(base64.b64decode(b))
                except Exception: t.write(b)
                p=t.name
            arr, sr = librosa.load(p, sr=16000, mono=True); os.unlink(p); return arr.astype(np.float32)
    if arr.ndim>1: arr=arr.mean(axis=1)
    if sr!=16000: arr=librosa.resample(arr, orig_sr=sr, target_sr=16000)
    return arr.astype(np.float32)

# décodeurs KenLM par langue
def unigrams(lang, top=50000):
    from collections import Counter
    c=Counter()
    with open(f"{LM}/{lang}.txt", encoding="utf-8") as f:
        for line in f: c.update(line.split())
    return [w for w,_ in c.most_common(top)]

decoders = {}
for lang in ["swa","kik","luo","kln","mas","som"]:
    decoders[lang] = build_ctcdecoder(labels, kenlm_model_path=f"{LM}/{lang}.bin",
                                      unigrams=unigrams(lang), alpha=0.5, beta=1.0)
print("décodeurs prêts:", list(decoders.keys()))

### 11.4 — Inférence complète v3 + KenLM → submission.csv

In [ ]:
import glob, os, time, csv, io, gc, base64, torch, numpy as np
import soundfile as sf, librosa
from datasets import load_dataset
from collections import Counter

TEST_ROOT = "/content/drive/MyDrive/afrivoices/test"
SUB_OUT   = "/content/drive/MyDrive/afrivoices/submission.csv"
LAT_OUT   = "/content/drive/MyDrive/afrivoices/latency_report.csv"
OUT_COL   = "transcription"
MAX_TOK   = 8 * 20.0

# decode_robuste doit être défini (sinon recharge la cellule précédente)
torch.cuda.empty_cache()
files = sorted(glob.glob(os.path.join(TEST_ROOT, "**", "*.parquet"), recursive=True))
test = load_dataset("parquet", data_files=files, split="train")
has_lang = "language" in test.column_names
print(len(test), "exemples")

print("Décodage durées + tri...")
items = []
for idx in range(len(test)):
    try: items.append((idx, len(decode_robuste(test[idx]["audio"]))/16000))
    except Exception: items.append((idx, -1.0))
items.sort(key=lambda x: x[1])

sub_rows, lat_rows = [], []
total_audio = total_proc = 0.0
n_skip = done = 0
i = 0
while i < len(items):
    longest = items[i][1]
    if longest <= 0:
        sub_rows.append({"id": test[items[i][0]]["id"], OUT_COL: ""}); n_skip+=1; i+=1; done+=1; continue
    bs = min(16, max(1, int(MAX_TOK // max(longest,1.0))))
    batch_idx = [items[k][0] for k in range(i, min(i+bs,len(items))) if items[k][1] > 0]
    i += bs
    arrays, ids, langs, durs = [], [], [], []
    for idx in batch_idx:
        row = test[idx]
        try: arr = decode_robuste(row["audio"])
        except Exception:
            sub_rows.append({"id": row["id"], OUT_COL: ""}); n_skip+=1; continue
        arrays.append(arr); ids.append(row["id"])
        langs.append(row["language"] if has_lang else "swa"); durs.append(len(arr)/16000)
    if not arrays: continue
    feats = processor.feature_extractor(arrays, sampling_rate=16000, return_tensors="pt", padding=True)
    feats = {k: v.to(device) for k,v in feats.items()}
    t0 = time.perf_counter()
    with torch.no_grad():
        logits = model(**feats).logits.cpu().numpy()    # (B, T, V)
    # décodage LM par langue, individuel
    texts = []
    for bi in range(len(arrays)):
        dec = decoders.get(langs[bi]) or next(iter(decoders.values()))
        texts.append(dec.decode(logits[bi]).strip())
    dt = time.perf_counter() - t0
    ba = sum(durs) or 1e-6
    for _id, txt, dur, lang in zip(ids, texts, durs, langs):
        sub_rows.append({"id": _id, OUT_COL: txt})
        sh = dt*(dur/ba)
        lat_rows.append({"id": _id, "language": lang, "audio_s": round(dur,3),
                         "proc_s": round(sh,4), "rtf": round(sh/max(dur,1e-6),3)})
    total_audio += sum(durs); total_proc += dt
    del feats
    done += len(batch_idx)
    if done % 2000 < bs:
        torch.cuda.empty_cache(); gc.collect()
        print(f"  {done}/{len(test)}")

with open(SUB_OUT,"w",newline="",encoding="utf-8") as f:
    w=csv.DictWriter(f,fieldnames=["id",OUT_COL]); w.writeheader(); w.writerows(sub_rows)
with open(LAT_OUT,"w",newline="",encoding="utf-8") as f:
    w=csv.DictWriter(f,fieldnames=["id","language","audio_s","proc_s","rtf"]); w.writeheader(); w.writerows(lat_rows)
print(f"\n{len(sub_rows)} lignes ({n_skip} vides). RTF {total_proc/max(total_audio,1e-6):.3f}")
print("langues:", dict(Counter(r['language'] for r in lat_rows)))

### 11.5 — Assemblage final → submission_final_kenlm_v3.csv

In [ ]:
import pandas as pd, glob

sub = pd.read_csv("/content/drive/MyDrive/afrivoices/submission.csv")
lat = pd.read_csv("/content/drive/MyDrive/afrivoices/latency_report.csv")

m = sub.merge(lat[["id","language"]], on="id", how="left")
if m["language"].isna().any():
    files = sorted(glob.glob("/content/drive/MyDrive/afrivoices/test/**/*.parquet", recursive=True))
    meta = pd.concat([pd.read_parquet(f, columns=["id","language"]) for f in files], ignore_index=True)
    m["language"] = m["language"].fillna(m["id"].map(dict(zip(meta["id"], meta["language"]))))

final = pd.DataFrame({"id": m["id"], "language": m["language"],
                      "transcription": m["transcription"].fillna("").astype(str)})
final["transcription"] = final["transcription"].str.replace(r"[\r\n]+", " ", regex=True).str.strip()
final.loc[final["transcription"]=="", "transcription"] = "_"

assert final["id"].nunique() == len(final)
assert final.columns.tolist() == ["id","language","transcription"]
assert final["language"].isna().sum() == 0
assert (final["transcription"].str.strip()=="").sum() == 0

OUT = "/content/drive/MyDrive/afrivoices/submission_final_kenlm_v3.csv"
final.to_csv(OUT, index=False)
print(f"OK -> {OUT}  ({len(final)} lignes)")
print(final["language"].value_counts())

## 12. Soumission

Uploade `submission_final_kenlm_v3.csv` sur Kaggle (Submit Prediction).
Compare au score v2+KenLM = **0.469**. WER interne v3 = 0.440 macro (tous langages
améliorés vs v2). Garde v2 ET v3 tant que v3 n'est pas confirmé sur le leaderboard.